# Arase MGF 64 Hzデータのスピントーンの削除補正の実装テスト (erg_mgf_spintone_rm.pro (Imajo et al., 2021)の再現)

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# MGFデータの保存

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', no_update=True, get_support_data=True)

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

B64_data_dsi    = pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_sgi_quality_flag   = pt.data_quants['erg_mgf_l2_quality_64hz'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_sgi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(63, 'ms'))
print(len(ds_B64_dsi_segs))

In [ ]:
ds_B64_dsi_seg0 = ds_B64_dsi_segs[0]
print(ds_B64_dsi_seg0)

In [ ]:
pt.tplot_names()

In [ ]:
da_mgf_spin_phase_deg   = pt.data_quants['erg_mgf_l2_spin_phase_64hz'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
print(da_mgf_spin_phase_deg)

In [ ]:
time_base_B64   = ds_B64_dsi_seg0.time
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=time_base_B64)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

# Imajo et al. (2021)の再現

In [ ]:
import numpy as np

def _segment_by_spin_phase(phase_rad, wrap_threshold=-np.pi):
    """
    スピン位相のラップ位置（2π→0）を検出してインデックス区間に分ける。
    phase_rad: 1D array [rad]
    return: list of (i_start, i_end)  (両端含む)
    """
    phase_rad = np.asarray(phase_rad)
    dphi = np.diff(phase_rad)
    # ラップを検出 (大きく負のジャンプ)
    wrap_idx = np.where(dphi < wrap_threshold)[0]

    segments = []
    start = 0
    if len(wrap_idx) == 0:
        # ラップが見つからなければ全体を1セグメントとして扱う
        segments.append((0, len(phase_rad)-1))
        return segments

    for wi in wrap_idx:
        end = wi
        segments.append((start, end))
        start = wi + 1
    # 最後の区間
    if start < len(phase_rad):
        segments.append((start, len(phase_rad)-1))

    return segments


def _fit_A_B_segment(B_seg, phase_seg):
    """
    1 セグメントのデータに対して B ≈ A cos(φ) + B sin(φ) をフィット。
    """
    phase_seg = np.asarray(phase_seg)
    B_seg = np.asarray(B_seg)

    X = np.vstack([np.cos(phase_seg), np.sin(phase_seg)]).T  # (N, 2)
    # 最小二乗解（normal equation と等価）
    coef, *_ = np.linalg.lstsq(X, B_seg, rcond=None)
    A, B = coef
    return A, B


def _interp_params_to_full_time(time, t_fit, A_fit, B_fit):
    """
    区間ごとに得た (t_fit, A_fit, B_fit) を
    全 time に線形補間して A(t), B(t) を返す。
    """
    time = np.asarray(time)
    t_fit = np.asarray(t_fit)
    A_fit = np.asarray(A_fit)
    B_fit = np.asarray(B_fit)

    # datetime64 → 秒 に変換
    t0 = time[0].astype('datetime64[ns]')
    t_all_sec = (time - t0) / np.timedelta64(1, 's')
    t_fit_sec = (t_fit - t0) / np.timedelta64(1, 's')

    A_all = np.interp(t_all_sec, t_fit_sec, A_fit)
    B_all = np.interp(t_all_sec, t_fit_sec, B_fit)
    return A_all, B_all


def remove_spintone_single_component(time, B, phase_rad,
                                     min_points=10, wrap_threshold=-np.pi):
    """
    単一成分用のスピントーン除去。
    time: np.datetime64[ns] などの 1D array
    B: 1D array, 磁場成分
    phase_rad: 1D array, スピン位相 [rad]
    min_points: 各セグメントに必要な最小データ点数
    """
    time = np.asarray(time)
    B = np.asarray(B)
    phase_rad = np.asarray(phase_rad)

    segments = _segment_by_spin_phase(phase_rad, wrap_threshold=wrap_threshold)

    t_fit_list = []
    A_fit_list = []
    B_fit_list = []

    for i_start, i_end in segments:
        idx = slice(i_start, i_end+1)
        if (i_end - i_start + 1) < min_points:
            continue

        B_seg = B[idx]
        phase_seg = phase_rad[idx]
        t_seg = time[idx]
        A, Bc = _fit_A_B_segment(B_seg, phase_seg)

        # このセグメントの「代表時刻」は中央インデックスの時間にしておく
        mid_idx = (i_start + i_end) // 2
        t_fit_list.append(time[mid_idx])
        A_fit_list.append(A)
        B_fit_list.append(Bc)

    if len(t_fit_list) < 2:
        raise RuntimeError("有効なスピン区間が足りず、A,B を補間できない。")

    t_fit = np.array(t_fit_list)
    A_fit = np.array(A_fit_list)
    B_fit = np.array(B_fit_list)

    # 全 time へ補間
    A_all, B_all = _interp_params_to_full_time(time, t_fit, A_fit, B_fit)

    # モデルスピントーン波形
    B_spt = A_all * np.cos(phase_rad) + B_all * np.sin(phase_rad)
    B_clean = B - B_spt

    return B_clean, B_spt, A_all, B_all, t_fit, A_fit, B_fit


In [ ]:
def remove_spintone_3comp(time, Bx, By, Bz, phase_rad, **kwargs):
    """
    Bx, By, Bz をまとめてスピントーン除去。
    kwargs は remove_spintone_single_component にそのまま渡す。
    """
    Bx_clean, Bx_spt, Ax_all, Bx_all, *_ = remove_spintone_single_component(
        time, Bx, phase_rad, **kwargs
    )
    By_clean, By_spt, Ay_all, By_all, *_ = remove_spintone_single_component(
        time, By, phase_rad, **kwargs
    )
    Bz_clean, Bz_spt, Az_all, Bz_all, *_ = remove_spintone_single_component(
        time, Bz, phase_rad, **kwargs
    )

    B_clean = np.vstack([Bx_clean, By_clean, Bz_clean]).T
    B_spt = np.vstack([Bx_spt, By_spt, Bz_spt]).T

    params = {
        "A_all": {"x": Ax_all, "y": Ay_all, "z": Az_all},
        "B_all": {"x": Bx_all, "y": By_all, "z": Bz_all},
    }

    return B_clean, B_spt, params


In [ ]:
time    = ds_B64_dsi_seg0.time.values
Bx      = ds_B64_dsi_seg0['B64_dsi_x'].values
By      = ds_B64_dsi_seg0['B64_dsi_y'].values
Bz      = ds_B64_dsi_seg0['B64_dsi_z'].values

B_clean, B_spt, params = remove_spintone_3comp(
    time, Bx, By, Bz, da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

In [ ]:
ds_B64_dsi_seg0_spt = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt[:, 2])
    }, coords={'time': time})

ds_B64_dsi_seg0_clean = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean[:, 2])
    }, coords={'time': time})

print(ds_B64_dsi_seg0_spt)
print(ds_B64_dsi_seg0_clean)

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

time_range_analysis     = ['20220901/22:37:00', '20220901/22:38:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

ds_B64_dsi_seg0_analysis        = ds_B64_dsi_seg0.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_B64_dsi_seg0_spt_analysis    = ds_B64_dsi_seg0_spt.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_B64_dsi_seg0_clean_analysis  = ds_B64_dsi_seg0_clean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

plt.figure(figsize=(10, 5))
plt.plot(ds_B64_dsi_seg0_analysis.time, ds_B64_dsi_seg0_analysis['B64_dsi_x'], label='raw', alpha=0.5)
plt.plot(ds_B64_dsi_seg0_clean_analysis.time, ds_B64_dsi_seg0_clean_analysis['B64_dsi_x_clean'], label='clean', lw=1)
plt.legend()
plt.minorticks_on()
plt.grid(True, alpha=0.5, which='both')

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
plt.xlim(time_range_T_analysis_0, time_range_T_analysis_1)

plt.show()

plt.figure(figsize=(10, 5))
plt.plot(ds_B64_dsi_seg0_spt_analysis.time, ds_B64_dsi_seg0_spt_analysis['B64_dsi_x_spt'], label='spin tone', lw=1)
plt.legend()
plt.minorticks_on()
plt.grid(True, alpha=0.5, which='both')

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
plt.xlim(time_range_T_analysis_0, time_range_T_analysis_1)

plt.show()

In [ ]:
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.erg_mgf_spintone_rm as emsr
importlib.reload(emsr)
os.chdir('./KAW_observation')
print(os.getcwd())

time    = ds_B64_dsi_seg0.time.values
Bx      = ds_B64_dsi_seg0['B64_dsi_x'].values
By      = ds_B64_dsi_seg0['B64_dsi_y'].values
Bz      = ds_B64_dsi_seg0['B64_dsi_z'].values

B_clean_func, B_spt_func, params_func = emsr.remove_spintone_3comp(
    time, Bx, By, Bz, da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_seg0_spt_func = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_func[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_func[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_func[:, 2])
    }, coords={'time': time})

ds_B64_dsi_seg0_clean_func = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_func[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_func[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_func[:, 2])
    }, coords={'time': time})

print(ds_B64_dsi_seg0_spt_func)
print(ds_B64_dsi_seg0_clean_func)

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

time_range_analysis     = ['20220901/22:35:00', '20220901/22:40:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

ds_B64_dsi_seg0_analysis        = ds_B64_dsi_seg0.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_B64_dsi_seg0_spt_analysis    = ds_B64_dsi_seg0_spt_func.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_B64_dsi_seg0_clean_analysis  = ds_B64_dsi_seg0_clean_func.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

plt.figure(figsize=(10, 5))
plt.plot(ds_B64_dsi_seg0_analysis.time, ds_B64_dsi_seg0_analysis['B64_dsi_x'], label='raw', alpha=0.5)
plt.plot(ds_B64_dsi_seg0_clean_analysis.time, ds_B64_dsi_seg0_clean_analysis['B64_dsi_x_clean'], label='clean', lw=1)
plt.legend()
plt.minorticks_on()
plt.grid(True, alpha=0.5, which='both')

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
plt.xlim(time_range_T_analysis_0, time_range_T_analysis_1)

plt.show()

plt.figure(figsize=(10, 5))
plt.plot(ds_B64_dsi_seg0_spt_analysis.time, ds_B64_dsi_seg0_spt_analysis['B64_dsi_x_spt'], label='spin tone', lw=1)
plt.legend()
plt.minorticks_on()
plt.grid(True, alpha=0.5, which='both')

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
plt.xlim(time_range_T_analysis_0, time_range_T_analysis_1)

plt.show()